# STOCK EVALUATION MODEL  

In [1]:
!pip install yfinance

     ------------------------------------ 149.3/149.3 kB 211.7 kB/s eta 0:00:00
     ---------------------------------------- 2.0/2.0 MB 53.7 kB/s eta 0:00:00
     ---------------------------------------- 4.0/4.0 MB 35.6 kB/s eta 0:00:00
     ------------------------------------- 193.9/193.9 kB 27.5 kB/s eta 0:00:00
     ------------------------------------- 456.5/456.5 kB 77.8 kB/s eta 0:00:00
     ------------------------------------ 217.8/217.8 kB 111.6 kB/s eta 0:00:00
     ------------------------------------- 185.1/185.1 kB 91.0 kB/s eta 0:00:00
  Attempting uninstall: cffi
    Found existing installation: cffi 1.17.1
    Uninstalling cffi-1.17.1:
      Successfully uninstalled cffi-1.17.1



[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: C:\Users\avnik\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd

## — DATA PIPELINE

Pulls the raw financial metrics needed to score a company across:
Revenue Growth, Profitability, Valuation, Debt, and Cash Flow.


I started by testing the model on Apple, Microsoft and NVIDIA because they are large technology companies with different business models and financial characteristics. The purpose of this first test was to see whether the data pipeline could collect the required information and whether the scoring model could turn those figures into comparable scores.

In [3]:
'''
This code Pulls the key raw metrics for one company from Yahoo Finance. Returns a flat dictionary — this is the raw material our scoring
model will later turn into a 0-100 investment score.
'''
# This creates a function called: get_company_metrics In this case, the task is: Get important financial metrics for a company.
# (ticker_symbol: str) The function expects a company ticker symbol as input. For example APPL. : str This is called a type hint. It tells us that ticker_symbol is expected to be a string.
# -> dict This is another type hint. It tells us that the function is expected to return a dictionary.
# ticker = yf.Ticker(ticker_symbol) This line uses yfinance to create a ticker object. For example, if: ticker_symbol = "AAPL"
# ticker.info This asks yfinance to retrieve available company information. The results are stored inside the variable: info

def get_company_metrics(ticker_symbol: str) -> dict:

    ticker = yf.Ticker(ticker_symbol)
    info = ticker.info


# This creates an empty dictionary structure called: metrics
# Why use .get()? The code uses: info.get("longName") instead of: info["longName"] because .get() is safer.
# If the information is missing, Python will usually return: None instead of producing an error.

    metrics = {
        "ticker": ticker_symbol,
        "name": info.get("longName"),
        "sector": info.get("sector"),
        "industry": info.get("industry"),
 
        # --- Growth ---
        "revenue_growth": info.get("revenueGrowth"),        # e.g. 0.08 = 8%
        "earnings_growth": info.get("earningsGrowth"),
 
        # --- Profitability ---
        "gross_margin": info.get("grossMargins"),
        "operating_margin": info.get("operatingMargins"),
        "net_margin": info.get("profitMargins"),
        "return_on_equity": info.get("returnOnEquity"),
 
        # --- Valuation ---
        "trailing_pe": info.get("trailingPE"),
        "forward_pe": info.get("forwardPE"),
        "price_to_book": info.get("priceToBook"),
        "market_cap": info.get("marketCap"),
 
        # --- Debt ---
        "debt_to_equity": info.get("debtToEquity"),
        "current_ratio": info.get("currentRatio"),
        "total_debt": info.get("totalDebt"),
        "total_cash": info.get("totalCash"),
 
        # --- Cash Flow ---
        "operating_cashflow": info.get("operatingCashflow"),
        "free_cashflow": info.get("freeCashflow"),
    }
 
#This sends the completed dictionary back to wherever the function was called. For example: data = get_company_metrics("AAPL")
# The function collects Apple's information and then returns it. That returned information is stored inside: data (Next line of code)
    return metrics


# if __name__ == "__main__": This is a very common Python pattern. It basically means: Only run the code underneath this section if this file is being run directly.
# For example, if you run: python data_pipeline.py then the code underneath will execute. However, if another Python file imports this file, the code underneath will not automatically run.

# apple = get_company_metrics("AAPL") This calls the function. The ticker: "AAPL" is passed into: get_company_metrics()
# The function then:
# 1.Connects to Yahoo Finance through yfinance.
# 2.Finds Apple.
# 3.Retrieves the available company information.
# 4.Selects the required financial metrics.
# 5.Stores them in a dictionary.
# 6.Returns the dictionary.
#The result is stored in: apple


if __name__ == "__main__":
    # Get financial metrics for each company
    apple = get_company_metrics("AAPL")
    microsoft = get_company_metrics("MSFT")
    nvidia = get_company_metrics("NVDA")

    # Put all three company dictionaries into a list
    companies = [apple, microsoft, nvidia]

    # Convert the list of dictionaries into a pandas DataFrame
    df = pd.DataFrame(companies)


In [4]:
df

,ticker,name,sector,industry,revenue_growth,earnings_growth,gross_margin,operating_margin,net_margin,return_on_equity,trailing_pe,forward_pe,price_to_book,market_cap,debt_to_equity,current_ratio,total_debt,total_cash,operating_cashflow,free_cashflow
0,AAPL,Apple Inc.,Technology,Consumer Electronics,0.164,0.287,0.48653,0.32623,0.27619,1.48751,38.210102,34.729748,45.218750,4857088966656,78.445,1.003,84343996416,62399000576,146723995648,107721875456
1,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,0.177,0.317,0.67944,0.45111,0.40305,0.34039,27.884226,21.221033,8.398272,3714577334272,29.118,1.230,128812998656,76842999808,182934994944,16545500160
2,NVDA,NVIDIA Corporation,Technology,Semiconductors,1.059,1.278,0.74674,0.66237,0.63663,1.17211,26.657671,13.486845,22.207699,5085251764224,16.971,4.589,38860001280,62469001216,134359998464,41809874944


The first results show that the model is able to collect the main financial metrics for all three companies. The companies can now be compared using the same five areas: revenue growth, profitability, valuation, debt and cash flow. This gives me a working starting point before testing the model on companies from other industries.

## SCORING ENGINE

The raw financial figures cannot be compared directly because they are measured on very different scales. For example, market capitalisation is measured in trillions of dollars, while a current ratio is a much smaller number. I therefore take the raw metrics dictionary produced by data_pipeline.py and convert it into a 0-100 investment score across five categories:

Revenue Growth | Profitability | Valuation | Debt | Cash Flow
 
v1 approach: equal weighting (20% each category). Each category is normalized
against reasonable benchmark ranges, since raw numbers (a P/E of 38 vs a
debt-to-equity of 78) aren't on comparable scales. This is deliberately a
transparent, rules-based model rather than a black box and every score can
be explained line by line, which matters for a finance audience.

This is important because a company doesn't get its entire score from just one metric.

**For example, Nvidia might have extremely strong growth, but the model still considers its valuation, debt and cash flow.**

The code also explains that the raw metrics are on different scales, so they need to be normalized before they can be combined.
 
Note: benchmark ranges are general-purpose starting points, not
sector-specific. A sector-relative version is a natural v2.

In [5]:
'''
Based on the rules we've established, how strong is this company in each category?

Revenue Growth | Profitability | Valuation | Debt | Cash Flow

Each category eventually contributes to the overall investment score.

'''

## clamp() This is a function created to Keep a score between 0 and 100. for example if clamp(75) output is 75 but if it is clamp(120) = 1-- or if it is clamp(-2) = 0
## return max(low, min(high, value)) Firstly: min(high, value) Suppose: high = 100 value = 120 Then: min(100, 120) returns: 100
## So it prevents the number from going above 100. Then: max(low, 100) makes sure it doesn't go below 0.
## So: 120 → 100 Likewise: -20 → 0 This is useful because some of the formulas can produce scores above 100 or below 0.

def clamp(value, low=0, high=100):
    """Keeps a score within 0-100, in case a metric is unusually extreme."""
    return max(low, min(high, value))
 
## a function called: score_revenue_growth() is created. It takes: metrics which is expected to be a dictionary.
## Remember the earlier data: revenue_growth: 0.164. The purpose of this function is to turn that into a 0–100 score.

## growth = metrics.get("revenue_growth"). This looks inside the dictionary for: "revenue_growth" For Apple, that was: 0.164 So: growth becomes: 0.164
## if growth is None: None basically means: There is no value available. So this asks: "Did we fail to get revenue growth?" If yes: return 50.0
## The model gives the company a neutral score of 50 rather than automatically treating missing information as terrible.
## return clamp((growth / 0.30) * 100) This is the actual scoring formula.
## For Apple: growth = 0.164 The calculation is: (0.164 / 0.30) × 100 which is approximately: 54.7.

def score_revenue_growth(metrics: dict) -> float:
    """
    0% growth -> 0 points. 30%+ growth -> 100 points (near max for a v1 scale).
    Linear in between. Missing data defaults to a neutral 50. In summary 0% growth = 0, 15% growth = 50, 30% growth = 100
    """
    growth = metrics.get("revenue_growth")
    if growth is None:
        return 50.0
    return clamp((growth / 0.30) * 100)
 
 
def score_profitability(metrics: dict) -> float:
    """
    Averages four profitability signals, each normalized against a
    "strong performance" benchmark:
      - Gross margin:      40%+ = full marks
      - Operating margin:  25%+ = full marks
      - Net margin:        20%+ = full marks
      - Return on equity:  25%+ = full marks (capped, since >100% ROE
                            usually reflects high leverage or buybacks
                            rather than pure operating strength)
    """
    gross = metrics.get("gross_margin") or 0
    operating = metrics.get("operating_margin") or 0
    net = metrics.get("net_margin") or 0
    roe = metrics.get("return_on_equity") or 0

## gross_score = clamp((gross / 0.40) * 100) The benchmark is: 40% gross margin = 100 points So if a company has:
## 40% → 100 points
## 20% → 50 points
## 10% → 25 points

## For example Apple has: 48.653% Because Apple is above the 40% benchmark, the formula initially produces more than 100. clamp() then brings it back down to:100.
## So Apple receives 100 points for gross margin.
 
    gross_score = clamp((gross / 0.40) * 100)
    operating_score = clamp((operating / 0.25) * 100)
    net_score = clamp((net / 0.20) * 100)

## roe_score = clamp((min(roe, 0.50) / 0.25) * 100) The model considers: 25% ROE = 100 points But it doesn't want extremely high ROE values to dominate the score.
## So it first does: min(roe, 0.50) That means: Use the ROE, but don't allow the input to exceed 50%.
## Apple's ROE is: 1.48751 which represents approximately: 148.75% The model therefore changes the input to: 0.50
## Then: (0.50 / 0.25) × 100 = 200 Then clamp() limits it to: 100 So Apple gets: ROE score = 100
## The reason for this cap is explained in the code: extremely high ROE can sometimes result from things such as high leverage or share buybacks rather than simply exceptional operating performance.

    roe_score = clamp((min(roe, 0.50) / 0.25) * 100)  # cap input at 50% ROE
 
    return (gross_score + operating_score + net_score + roe_score) / 4
 
 
def score_valuation(metrics: dict) -> float:
    """
    Lower valuation multiples score higher (cheaper relative to
    fundamentals = more attractive on a value basis).
    
      The model's benchmarks are:
      P/E ≤ 15      → 100 points
      P/E ≥ 45      → 0 points
      
      P/B ≤ 2       → 100 points
      P/B ≥ 15      → 0 points
      
    Averaged across both. This intentionally does NOT judge whether a
    premium is "justified" (e.g. by growth) — that nuance belongs in your
    own read of the results, not the raw score.
    Lower valuation multiples receive higher scores.   
    """
    pe = metrics.get("trailing_pe")
    pb = metrics.get("price_to_book")
 
    pe_score = 50.0 if pe is None else clamp(100 - ((pe - 15) / 30) * 100)
    pb_score = 50.0 if pb is None else clamp(100 - ((pb - 2) / 13) * 100)
 
    return (pe_score + pb_score) / 2
 
 
def score_debt(metrics: dict) -> float:
    """
    Lower debt-to-equity and healthier current ratio score higher.
      - Debt/Equity: 0 = 100 points, 100+ = 0 points
      - Current ratio: 2.0+ = 100 points, 0.5 or below = 0 points
    """
    dte = metrics.get("debt_to_equity")
    current = metrics.get("current_ratio")
 
    dte_score = 50.0 if dte is None else clamp(100 - (dte / 100) * 100)
    current_score = 50.0 if current is None else clamp(((current - 0.5) / 1.5) * 100)
 
    return (dte_score + current_score) / 2
 
 
def score_cash_flow(metrics: dict) -> float:
    """
    Two signals, averaged:
      - Cash yield: free cash flow as a % of market cap.
        3%+ yield = 100 points, 0% or negative = 0 points.
      - FCF conversion quality: free cash flow / operating cash flow.
        High conversion (close to or above 1.0) suggests earnings are
        backed by real cash, not just accounting profit. 70%+ = full marks.
    """
    fcf = metrics.get("free_cashflow")
    ocf = metrics.get("operating_cashflow")
    market_cap = metrics.get("market_cap")
 
    if fcf is None or market_cap in (None, 0):
        yield_score = 50.0
    else:
        cash_yield = fcf / market_cap
        yield_score = clamp((cash_yield / 0.03) * 100)
 
    if fcf is None or ocf in (None, 0):
        conversion_score = 50.0
    else:
        conversion = fcf / ocf
        conversion_score = clamp((conversion / 0.70) * 100)
 
    return (yield_score + conversion_score) / 2
 
 
def calculate_investment_score(metrics: dict, weights: dict = None) -> dict:
    """
    Combines all five category scores into one overall investment score.
    Default weighting is equal (20% each) for v1.
    """
    if weights is None:
        weights = {
            "revenue_growth": 0.20,
            "profitability": 0.20,
            "valuation": 0.20,
            "debt": 0.20,
            "cash_flow": 0.20,
        }
 
    category_scores = {
        "revenue_growth": score_revenue_growth(metrics),
        "profitability": score_profitability(metrics),
        "valuation": score_valuation(metrics),
        "debt": score_debt(metrics),
        "cash_flow": score_cash_flow(metrics),
    }
 
    overall = sum(category_scores[cat] * weights[cat] for cat in category_scores)
 
    return {
        "ticker": metrics.get("ticker"),
        "name": metrics.get("name"),
        "category_scores": {k: round(v, 1) for k, v in category_scores.items()},
        "overall_score": round(overall, 1),
    }
 
 
if __name__ == "__main__":
    # Real data pulled from data_pipeline.py for AAPL, MSFT, NVDA
    sample_companies = [
        {
            "ticker": "AAPL", "name": "Apple Inc.",
            "revenue_growth": 0.164, "earnings_growth": 0.287,
            "gross_margin": 0.48653, "operating_margin": 0.32623,
            "net_margin": 0.27619, "return_on_equity": 1.48751,
            "trailing_pe": 38.148106, "forward_pe": 34.703167,
            "price_to_book": 45.145380, "market_cap": 4849207869440,
            "debt_to_equity": 78.445, "current_ratio": 1.003,
            "total_debt": 84343996416, "total_cash": 62399000576,
            "operating_cashflow": 146723995648, "free_cashflow": 107721875456,
        },
        {
            "ticker": "MSFT", "name": "Microsoft Corporation",
            "revenue_growth": 0.177, "earnings_growth": 0.317,
            "gross_margin": 0.67944, "operating_margin": 0.45111,
            "net_margin": 0.40305, "return_on_equity": 0.34039,
            "trailing_pe": 27.627090, "forward_pe": 21.025343,
            "price_to_book": 8.320827, "market_cap": 3680323239936,
            "debt_to_equity": 29.118, "current_ratio": 1.230,
            "total_debt": 128812998656, "total_cash": 76842999808,
            "operating_cashflow": 182934994944, "free_cashflow": 16545500160,
        },
        {
            "ticker": "NVDA", "name": "NVIDIA Corporation",
            "revenue_growth": 1.059, "earnings_growth": 1.278,
            "gross_margin": 0.74674, "operating_margin": 0.66237,
            "net_margin": 0.63663, "return_on_equity": 1.17211,
            "trailing_pe": 27.631645, "forward_pe": 14.022135,
            "price_to_book": 23.019087, "market_cap": 5271048421376,
            "debt_to_equity": 16.971, "current_ratio": 4.589,
            "total_debt": 38860001280, "total_cash": 62469001216,
            "operating_cashflow": 134359998464, "free_cashflow": 41809874944,
        },
    ]
 
        # Calculate the scores for each company
    results = []

    for company in sample_companies:
        result = calculate_investment_score(company)

        # Create one row for each company
        results.append({
            "Ticker": result["ticker"],
            "Company": result["name"],
            "Revenue Growth": result["category_scores"]["revenue_growth"],
            "Profitability": result["category_scores"]["profitability"],
            "Valuation": result["category_scores"]["valuation"],
            "Debt": result["category_scores"]["debt"],
            "Cash Flow": result["category_scores"]["cash_flow"],
            "Overall Score": result["overall_score"]
        })

    # Convert the results into a pandas DataFrame
    results_df = pd.DataFrame(results)

    # Display the table
    print(results_df.to_string(index=False))

Ticker               Company  Revenue Growth  Profitability  Valuation  Debt  Cash Flow  Overall Score
  AAPL            Apple Inc.            54.7          100.0       11.4  27.5       87.0           56.1
  MSFT Microsoft Corporation            59.0          100.0       54.6  59.8       14.0           57.5
  NVDA    NVIDIA Corporation           100.0          100.0       28.9  91.5       35.4           71.2


The model uses five categories and gives each category a 20 percent weighting. This means that a company is not judged on one measure alone. A company can have strong growth but still receive a lower overall score if its valuation, debt or cash flow is weaker.

Looking at the scores, I did not want to assume that the company with the highest overall score was automatically the best investment. Instead, I wanted to understand what was driving each score. This is important because the overall score is made up of several different categories, so a high score in one area can offset a weaker result in another.

NVIDIA has the highest overall score at 71.2. Its strongest areas are revenue growth, profitability and debt. Apple and Microsoft have lower overall scores, but for different reasons. Apple receives a much lower valuation score, while Microsoft receives a much lower cash flow score. This shows why looking at the individual category scores is useful rather than looking only at the final number.

## - BATCH RUNNER

After getting the model working on Apple, Microsoft and NVIDIA, I wanted to see whether the same scoring system would work across different industries. I therefore added companies from financial services, healthcare, consumer staples, energy, telecommunications and retail.

The purpose of this test was not to find the best company. I wanted to see whether the model behaved consistently when companies had very different financial structures.

In [6]:
# The functions get_company_metrics() and calculate_investment_score()
# have already been created earlier in this notebook.


TICKERS = [
    "AAPL",   # Tech - consumer electronics
    "MSFT",   # Tech - software
    "NVDA",   # Tech - semiconductors
    "JPM",    # Financials - bank (naturally high leverage)
    "JNJ",    # Healthcare - pharma/consumer health
    "PG",     # Consumer staples - stable, slow growth
    "XOM",    # Energy - cyclical, capital intensive
    "KO",     # Consumer staples - stable dividend payer
    "T",      # Telecom - high debt, low growth, high yield
    "COST",   # Retail - low margin, high volume
]
 
 
def run_batch(tickers: list) -> pd.DataFrame:
    rows = []
    for ticker in tickers:
        try:
            metrics = get_company_metrics(ticker)
            result = calculate_investment_score(metrics)
            row = {
                "ticker": result["ticker"],
                "name": result["name"],
                "overall_score": result["overall_score"],
                **result["category_scores"],
            }
            rows.append(row)
        except Exception as e:
            print(f"Skipped {ticker}: {e}")
 
    df = pd.DataFrame(rows)
    df = df.sort_values("overall_score", ascending=False).reset_index(drop=True)
    return df
 
 
if __name__ == "__main__":
    results_df = run_batch(TICKERS)
    print(results_df.to_string(index=False))
 
    # Uncomment to save results for your GitHub repo / README screenshots
    # results_df.to_csv("investment_scores.csv", index=False)

ticker                            name  overall_score  revenue_growth  profitability  valuation  debt  cash_flow
   XOM ExxonMobil Holdings Corporation           76.5           100.0           58.4       86.4  63.2       74.6
   JPM            JPMorgan Chase & Co.           73.0           100.0           67.8       97.2  50.0       50.0
  NVDA              NVIDIA Corporation           71.6           100.0          100.0       30.6  91.5       35.9
   JNJ               Johnson & Johnson           60.2            22.0          100.0       51.3  40.8       87.1
    PG    The Procter & Gamble Company           58.7             5.0           95.1       71.2  23.7       98.5
  MSFT           Microsoft Corporation           57.3            59.0          100.0       54.0  59.8       13.9
     T                       AT&T Inc.           56.2             7.7           89.3      100.0  15.7       68.2
  AAPL                      Apple Inc.           56.1            54.7          100.0       11.3 

The wider industry test showed me that the model does not treat every industry in the same way. JPMorgan was an important example because some of the information required by the model was not available in the same way as it was for the technology companies.

I had to be careful here because a missing value does not necessarily mean that the company has poor financial performance. The scoring model uses 50 as a neutral score when certain information is missing. This means I could not treat the resulting score as a fully measured score.

JPMorgan received 50 for the affected categories because the model could not obtain the required values. These 50 point scores should therefore not be interpreted as JPMorgan actually performing at the middle of the scale. They represent missing information being given a neutral placeholder.

This is important because the overall score could give the impression that all five categories were measured when they were not. For this reason, I decided that comparing banks directly with technology companies using the same rules would not be a fair comparison.

### The problem with comparing different industries


The JPMorgan result made me look more closely at another issue. Different industries naturally have different financial structures. A bank, for example, cannot be assessed using exactly the same interpretation of debt and cash flow as a technology company.

I also saw a similar issue when looking at companies such as AT&T. A high level of debt can have a very different meaning for a telecommunications company because the industry requires large amounts of investment in infrastructure.

This showed me that the model can technically produce a score for companies from different industries, but that does not necessarily mean the scores are directly comparable. The problem is not that the companies are performing badly. The problem is that the same benchmark can mean different things in different industries.

Now that I have tested the model across different industries and identified the limitations of using one scoring system for every sector, I can move to the final comparison. The next step is to collect the latest available financial metrics for the five selected technology companies and run them through the same scoring model.

From this point onwards, I will focus on what the scores are showing, what is driving the differences between companies and whether the overall ranking makes sense when the individual categories are considered.

### Decision to focus on technology companies

**Why I decided to focus on technology stocks**

After testing the model across different industries, I decided to focus the final comparison on five large technology companies. The main reason was consistency. The technology companies provided a more comparable set of financial information, which makes it easier to apply the same scoring rules across all companies.

This also gives the project a clearer scope. Instead of trying to create one scoring system that treats every industry in the same way, I can focus on comparing companies that operate in a similar broad sector.

This does not mean that technology companies are automatically better investments than companies in other industries. It simply means that they provide a more consistent group for testing the current version of the model.

The wider industry test was therefore still useful. It showed me a limitation of the model before I made the final decision about the companies to include. Rather than ignoring that problem, I used the result of the test to narrow the scope of the research.

**Choosing the final five**

* Apple (AAPL)
* Microsoft (MSFT)
* NVIDIA (NVDA)
* Alphabet (GOOGL)
* Amazon (AMZN)

I selected Apple, Microsoft, NVIDIA, Alphabet and Amazon for the final comparison. These companies provide a useful mix within the technology sector, rather than all having exactly the same business model.

I also wanted to keep the number of companies manageable. Five companies provide enough data for comparison while keeping the analysis focused on the factors that are driving the investment scores.

The final group gives the model a consistent set of companies to compare using the same five categories. Amazon is also useful because its business model is different from companies such as Microsoft and NVIDIA, so its results can show whether the scoring model responds differently to lower margins and a different business mix.

## FINAL BATCH

The batch runner collects the financial data for each company, applies the scoring model and sorts the companies from the highest overall score to the lowest. The model gives each of the five categories an equal 20% weighting, so the overall score reflects revenue growth, profitability, valuation, debt and cash flow.

In [8]:
# Scoped to five large technology companies.
# Keeping the companies within one industry makes the results
# more comparable and reduces some of the sector differences
# seen during the wider industry test.

TICKERS = [
    "AAPL",   # Apple - consumer electronics
    "MSFT",   # Microsoft - software/cloud
    "NVDA",   # NVIDIA - semiconductors
    "GOOGL",  # Alphabet - search/cloud/advertising
    "AMZN",   # Amazon - e-commerce/cloud (lower margin profile than the other four)
]
 
 
def run_batch(tickers: list) -> pd.DataFrame:
    rows = []
    for ticker in tickers:
        try:
            metrics = get_company_metrics(ticker)
            result = calculate_investment_score(metrics)
            row = {
                "ticker": result["ticker"],
                "name": result["name"],
                "overall_score": result["overall_score"],
                **result["category_scores"],
            }
            rows.append(row)
        except Exception as e:
            print(f"Skipped {ticker}: {e}")
 
    df = pd.DataFrame(rows)
    df = df.sort_values("overall_score", ascending=False).reset_index(drop=True)
    return df
 
 
if __name__ == "__main__":
    results_df = run_batch(TICKERS)
    print(results_df.to_string(index=False))

ticker                  name  overall_score  revenue_growth  profitability  valuation  debt  cash_flow
 GOOGL         Alphabet Inc.           73.4            80.7          100.0       77.9  90.6       17.7
  NVDA    NVIDIA Corporation           71.6           100.0          100.0       30.6  91.5       35.9
  MSFT Microsoft Corporation           57.3            59.0          100.0       53.9  59.8       13.9
  AAPL            Apple Inc.           56.1            54.7          100.0       11.4  27.5       87.0
  AMZN      Amazon.com, Inc.           55.8            65.3           85.5       80.0  45.0        3.4


Alphabet received the highest overall score at 73.4. Its strongest results were profitability, debt and valuation. However, its cash flow score was much lower at 17.7. This means the overall score is not being driven by one measure alone. The company performs strongly in several categories, but it also has a clear weakness in cash flow under the current scoring rules.

NVIDIA ranked second with a score of 71.6. It received the maximum score for both revenue growth and profitability, as well as a strong debt score of 91.5. Its main weakness was valuation, where it scored 30.6. This shows that the model is recognising NVIDIA's strong growth and profitability while also reducing its score because of its valuation.

Microsoft scored 57.3. Its profitability score was 100, but its cash flow score was only 13.9. Its valuation and debt scores were also lower than Alphabet and NVIDIA. This suggests that Microsoft's strong profitability is being offset by weaker results in some of the other categories.

Apple scored 56.1. Its strongest category was cash flow at 86.9, while its valuation score was only 11.3 and its debt score was 27.5. This is a good example of why the overall score needs to be looked at together with the individual category scores. Apple performs strongly in cash flow, but the model gives it much lower scores for valuation and debt.

Amazon had the lowest overall score at 55.8. It performed well in revenue growth and valuation, but its profitability score was lower than the other four companies and its cash flow score was only 3.4. The result reflects the current scoring rules and shows that Amazon's strengths in growth and valuation are not enough to offset its weaker profitability and cash flow scores.

Overall, the results show that the model produces different investment profiles rather than simply ranking companies based on growth. The next step is therefore to look more closely at which financial metrics are driving these category scores and whether the ranking makes sense when the individual results are considered.